In [1]:
# Import required libraries
import boto3
from lab_helpers.utils import get_ssm_parameter
from lab_helpers.lab2_memory import create_or_get_memory_resource

create_or_get_memory_resource()  # Just in case the memory lab wasn't executed

'CustomerSupportMemory-X72O7a3jPa'

In [2]:
%%writefile ./lab_helpers/lab4_runtime.py
from bedrock_agentcore.runtime import (
    BedrockAgentCoreApp,
)  #### AGENTCORE RUNTIME - LINE 1 ####
from strands import Agent
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamable_http_client
import httpx   
import requests
import boto3
from strands.models import BedrockModel
from lab_helpers.utils import get_ssm_parameter
from lab_helpers.lab1_strands_agent import (
    get_return_policy,
    get_product_info,
    get_technical_support,
    SYSTEM_PROMPT,
    MODEL_ID,
)

from lab_helpers.lab2_memory import (
    CustomerSupportMemoryHooks,
    memory_client,
    ACTOR_ID,
    SESSION_ID,
)

# Initialize boto3 client
sts_client = boto3.client('sts')

# Get AWS account details
REGION = boto3.session.Session().region_name

# Lab1 import: Create the Bedrock model
model = BedrockModel(model_id=MODEL_ID)

# Lab2 import : Initialize memory via hooks
memory_id = get_ssm_parameter("/app/customersupport/agentcore/memory_id")
memory_hooks = CustomerSupportMemoryHooks(
    memory_id, memory_client, ACTOR_ID, SESSION_ID
)

# Initialize the AgentCore Runtime App
app = BedrockAgentCoreApp()  #### AGENTCORE RUNTIME - LINE 2 ####

@app.entrypoint  #### AGENTCORE RUNTIME - LINE 3 ####
async def invoke(payload, context=None):
    """AgentCore Runtime entrypoint function"""
    user_input = payload.get("prompt", "")

    # Access request headers - handle None case
    request_headers = context.request_headers or {}

    # Get Client JWT token
    auth_header = request_headers.get('Authorization', '')

    print(f"Authorization header: {auth_header}")
    # Get Gateway ID
    existing_gateway_id = get_ssm_parameter("/app/customersupport/agentcore/gateway_id")
    
    # Initialize Bedrock AgentCore Control client
    gateway_client = boto3.client(
        "bedrock-agentcore-control",
        region_name=REGION,
    )
    # Get existing gateway details
    gateway_response = gateway_client.get_gateway(gatewayIdentifier=existing_gateway_id)

    # Get gateway url
    gateway_url = gateway_response['gatewayUrl']

    # Create MCP client and agent within context manager if JWT token available
    if gateway_url and auth_header:
        try:
                mcp_client = MCPClient(lambda: streamable_http_client(
                    url=gateway_url,
                    http_client=httpx.AsyncClient(headers={"Authorization": auth_header})
                ))
                
                with mcp_client:
                    #tools = mcp_client.list_tools_sync()
                    tools = (
                        [
                            get_product_info,
                            get_return_policy,
                            get_technical_support
                        ]
                        + mcp_client.list_tools_sync()
                    )
                    
                    # Create the agent with all customer support tools
                    agent = Agent(
                        model=model,
                        tools=tools,
                        system_prompt=SYSTEM_PROMPT,
                        hooks=[memory_hooks],
                    )
                    # Invoke the agent
                    response = agent(user_input)
                    return response.message["content"][0]["text"]
        except Exception as e:
                print(f"MCP client error: {str(e)}")
                return f"Error: {str(e)}"
    else:
        return "Error: Missing gateway URL or authorization header"

if __name__ == "__main__":
    app.run()  #### AGENTCORE RUNTIME - LINE 4 ####

Overwriting ./lab_helpers/lab4_runtime.py


In [3]:
from lab_helpers.utils import get_or_create_cognito_pool

access_token = get_or_create_cognito_pool(refresh_token=True)
print(f"Access token: {access_token['bearer_token']}")

Access token: eyJraWQiOiJPVzFNbjhvbkdSQlovR0FIbDl2UWJsV1lGY0FLQ2psa3BmaGZ1alU2SzF3PSIsImFsZyI6IlJTMjU2In0.eyJzdWIiOiJjODYxZDMzMC0zMDIxLTcwZDUtYmVkNS02NjMyZmQyZjdmZDAiLCJpc3MiOiJodHRwczovL2NvZ25pdG8taWRwLnVzLXdlc3QtMi5hbWF6b25hd3MuY29tL3VzLXdlc3QtMl8wdkJtdjY1dzQiLCJjbGllbnRfaWQiOiI0aXJpaWdwcTZoOXQzM25xbXBvZWRrNTlqNiIsIm9yaWdpbl9qdGkiOiJjNTIwMTllNi05NGY1LTQwZjEtODAzNC00YjQxM2Q2ZDI2OGYiLCJldmVudF9pZCI6IjliZWQ2ZmFhLTI5NGYtNDIxMS05MmMxLThhZTU4MmM4NWM0YSIsInRva2VuX3VzZSI6ImFjY2VzcyIsInNjb3BlIjoiYXdzLmNvZ25pdG8uc2lnbmluLnVzZXIuYWRtaW4iLCJhdXRoX3RpbWUiOjE3OTI1MTI5NTQsImV4cCI6MTc5MjUxNjU1NCwiaWF0IjoxNzkyNTEyOTU0LCJqdGkiOiJhOWJiNjgxNC00YWNkLTRkZDgtYjJiZC1lZjY2ODBiYWM1NGUiLCJ1c2VybmFtZSI6InRlc3R1c2VyIn0.THnKeMjQokcgBoZzb-YX2Omh68INSQB2I_VcHgIGzsLGq44lPkprR1qNfhvPGuxP2GEh-oLAxAF5Ip8jX5JHtF8Ona-yqfRQVtijldYUdtCUsGXVwytDDG_bCqtPPrlu9FQDBle0wgfW5tym9k3W6gYLSJ4uD7L2TqH1DWfVxtUUNWtbLLElvuKjEI9bTfe2mmo5QWhy3dNWqwTWT9A0uu85xnAIZ0I7Isl7NZl23O3nMhlBHbJk84VNic29ICzTQJEpQ1G4BrV1CSefseTH-7cVQ-e6z-mN6UsJXl3ZQ6

In [4]:
from bedrock_agentcore_starter_toolkit import Runtime
from lab_helpers.utils import create_agentcore_runtime_execution_role

# Initialize the runtime toolkit
boto_session = boto3.session.Session()
region = boto_session.region_name

execution_role_arn = create_agentcore_runtime_execution_role()

agentcore_runtime = Runtime()

# Configure the deployment
response = agentcore_runtime.configure(
    entrypoint="lab_helpers/lab4_runtime.py",
    execution_role=execution_role_arn,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="customer_support_agent",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "allowedClients": [
                get_ssm_parameter("/app/customersupport/agentcore/client_id")
            ],
            "discoveryUrl": get_ssm_parameter(
                "/app/customersupport/agentcore/cognito_discovery_url"
            ),
        }
    },
    # Add custom header allowlist for Authorization and custom headers
    request_header_configuration={
        "requestHeaderAllowlist": [
            "Authorization",  # Required for OAuth propogation
            "X-Amzn-Bedrock-AgentCore-Runtime-Custom-H1",  # Custom header
        ]
    },
)

print("Configuration completed:", response)

✅ Created IAM role: CustomerSupportAssistantBedrockAgentCoreRole-us-west-2
Role ARN: arn:aws:iam::956723944867:role/CustomerSupportAssistantBedrockAgentCoreRole-us-west-2
✅ Created policy: CustomerSupportAssistantBedrockAgentCorePolicy-us-west-2
✅ Attached policy to role
Policy ARN: arn:aws:iam::956723944867:policy/CustomerSupportAssistantBedrockAgentCorePolicy-us-west-2


Entrypoint parsed: file=/home/sagemaker-user/amazon-bedrock-agentcore/lab_helpers/lab4_runtime.py, bedrock_agentcore_name=lab4_runtime
Configuring BedrockAgentCore agent: customer_support_agent


💡 No container engine found (Docker/Finch/Podman not installed)

✓ Default deployment uses CodeBuild (no container engine needed)

For local builds, install Docker, Finch, or Podman

Will create new memory with mode: STM_ONLY
Memory configuration: Short-term memory only


⚠️  [WARNING] Platform mismatch: Current system is 'linux/amd64' but Bedrock AgentCore requires 'linux/arm64'.
For deployment options and workarounds, see: 
https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html

Generated .dockerignore
Generated Dockerfile: /home/sagemaker-user/amazon-bedrock-agentcore/Dockerfile
Generated .dockerignore: /home/sagemaker-user/amazon-bedrock-agentcore/.dockerignore
Setting 'customer_support_agent' as default agent
Bedrock AgentCore configured: /home/sagemaker-user/amazon-bedrock-agentcore/.bedrock_agentcore.yaml


Configuration completed: config_path=PosixPath('/home/sagemaker-user/amazon-bedrock-agentcore/.bedrock_agentcore.yaml') dockerfile_path=PosixPath('/home/sagemaker-user/amazon-bedrock-agentcore/Dockerfile') dockerignore_path=PosixPath('/home/sagemaker-user/amazon-bedrock-agentcore/.dockerignore') runtime='None' region='us-west-2' account_id='956723944867' execution_role='arn:aws:iam::956723944867:role/CustomerSupportAssistantBedrockAgentCoreRole-us-west-2' ecr_repository=None auto_create_ecr=True memory_id=None


In [5]:
# Launch the agent (this will build and deploy the container)
from lab_helpers.utils import put_ssm_parameter

launch_result = agentcore_runtime.launch()
print("Launch completed:", launch_result.agent_arn)

agent_arn = put_ssm_parameter(
    "/app/customersupport/agentcore/runtime_arn", launch_result.agent_arn
)

🚀 CodeBuild mode: building in cloud (RECOMMENDED - DEFAULT)
   • Build ARM64 containers in the cloud with CodeBuild
   • No local Docker required
💡 Available deployment modes:
   • runtime.launch()                           → CodeBuild (current)
   • runtime.launch(local=True)                 → Local development
   • runtime.launch(local_build=True)           → Local build + cloud deploy (NEW)
Creating memory resource for agent: customer_support_agent
✅ MemoryManager initialized for region: us-west-2
Creating new STM-only memory...
Created memory: customer_support_agent_mem-Vott2YApsy
✅ New memory created: customer_support_agent_mem-Vott2YApsy (provisioning in background)
Starting CodeBuild ARM64 deployment for agent 'customer_support_agent' to account 956723944867 (us-west-2)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: customer_support_agent
✅ ECR repository available: 956723944867.dkr.ecr.us-west-2.amazonaws.com/bedrock-

Repository doesn't exist, creating new ECR repository: bedrock-agentcore-customer_support_agent


Getting or creating CodeBuild execution role for agent: customer_support_agent
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-aa96c3a063
CodeBuild role doesn't exist, creating new role: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-aa96c3a063
Creating IAM role: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-aa96c3a063
✓ Role created: arn:aws:iam::956723944867:role/AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-aa96c3a063
Attaching inline policy: CodeBuildExecutionPolicy to role: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-aa96c3a063
✓ Policy attached: CodeBuildExecutionPolicy
Waiting for IAM role propagation...
CodeBuild execution role creation complete: arn:aws:iam::956723944867:role/AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-aa96c3a063
Created S3 bucket: bedrock-agentcore-codebuild-sources-956723944867-us-west-2
Using dockerignore.template with 45 patterns for zip filtering
Uploaded source to S3: customer_support_agent/source.zip
Created CodeBuild project: bedrock-agentcore-cu

Launch completed: arn:aws:bedrock-agentcore:us-west-2:956723944867:runtime/customer_support_agent-SOgLnLFPkn


In [6]:
# Check current status
status_response = agentcore_runtime.status()

if status_response.endpoint is None:
    print("Launching agent...")
    agentcore_runtime.launch()  # Try launch() instead of deploy()
    print("✅ Launch initiated!")
else:
    print(f"Already deployed. Status: {status_response.endpoint['status']}")
    

✅ MemoryManager initialized for region: us-west-2
🔎 Retrieving memory resource with ID: customer_support_agent_mem-Vott2YApsy...
  ✅ Found memory: customer_support_agent_mem-Vott2YApsy
Retrieved Bedrock AgentCore status for: customer_support_agent


Already deployed. Status: READY


In [7]:
import time

# Wait for the agent to be ready
status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]

end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    print(f"Waiting for deployment... Current status: {status}")
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]

print(f"Final status: {status}")

✅ MemoryManager initialized for region: us-west-2
🔎 Retrieving memory resource with ID: customer_support_agent_mem-Vott2YApsy...
  ✅ Found memory: customer_support_agent_mem-Vott2YApsy
Retrieved Bedrock AgentCore status for: customer_support_agent


Final status: READY


In [8]:
# Initialize the AgentCore Control client
client = boto3.client("bedrock-agentcore-control")

# Get agent ARN from status response
status_response = agentcore_runtime.status()

# Check what's available and extract runtime ID
if status_response.endpoint is None:
    print("❌ Agent not deployed yet. Run: agentcore_runtime.launch()")
else:
    print(f"Available keys: {status_response.endpoint.keys()}")
    # Try different possible key names
    agent_arn = (status_response.endpoint.get('endpointArn') or 
                 status_response.endpoint.get('agentArn') or 
                 status_response.endpoint.get('arn'))
    
    if agent_arn:
        runtime_id = agent_arn.split(":")[-1].split("/")[-1]
        print(f"Runtime ID: {runtime_id}")
    else:
        print(f"Full endpoint data: {status_response.endpoint}")

✅ MemoryManager initialized for region: us-west-2
🔎 Retrieving memory resource with ID: customer_support_agent_mem-Vott2YApsy...
  ✅ Found memory: customer_support_agent_mem-Vott2YApsy
Retrieved Bedrock AgentCore status for: customer_support_agent


Available keys: dict_keys(['ResponseMetadata', 'liveVersion', 'agentRuntimeEndpointArn', 'agentRuntimeArn', 'status', 'createdAt', 'lastUpdatedAt', 'name', 'id'])
Full endpoint data: {'ResponseMetadata': {'RequestId': 'e81decd0-8e1e-4cb0-820a-4150f27e31df', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Tue, 20 Oct 2026 16:39:48 GMT', 'content-type': 'application/json', 'content-length': '417', 'connection': 'keep-alive', 'x-amzn-requestid': 'e81decd0-8e1e-4cb0-820a-4150f27e31df', 'x-amz-apigw-id': 'FjTlUEMmPHcEb4A=', 'x-amzn-trace-id': 'Root=1-6ad79954-606d2ff02811e6c806986481'}, 'RetryAttempts': 0}, 'liveVersion': '1', 'agentRuntimeEndpointArn': 'arn:aws:bedrock-agentcore:us-west-2:956723944867:runtime/customer_support_agent-SOgLnLFPkn/runtime-endpoint/DEFAULT', 'agentRuntimeArn': 'arn:aws:bedrock-agentcore:us-west-2:956723944867:runtime/customer_support_agent-SOgLnLFPkn', 'status': 'READY', 'createdAt': datetime.datetime(2026, 10, 20, 16, 32, 24, 279059, tzinfo=tzlocal()), 'lastUpd

In [9]:
# Initialize the AgentCore Control client
client = boto3.client("bedrock-agentcore-control")

# Get agent ARN from status response
status_response = agentcore_runtime.status()
agent_arn = status_response.endpoint['agentRuntimeArn']

# Extract runtime ID from the ARN (format: arn:aws:bedrock-agentcore:region:account:runtime/runtime-id)
runtime_id = agent_arn.split(":")[-1].split("/")[-1]
print(f"Runtime ID: {runtime_id}")

✅ MemoryManager initialized for region: us-west-2
🔎 Retrieving memory resource with ID: customer_support_agent_mem-Vott2YApsy...
  ✅ Found memory: customer_support_agent_mem-Vott2YApsy
Retrieved Bedrock AgentCore status for: customer_support_agent


Runtime ID: customer_support_agent-SOgLnLFPkn


In [10]:
import lab_helpers.utils as utils
import inspect

print("Available functions:")
for name, func in inspect.getmembers(utils, inspect.isfunction):
    if not name.startswith('_'):
        print(f"  - {name}")

Available functions:
  - agentcore_memory_cleanup
  - cleanup_cognito_resources
  - create_agentcore_runtime_execution_role
  - delete_agentcore_runtime_execution_role
  - delete_customer_support_secret
  - delete_observability_resources
  - delete_ssm_parameter
  - gateway_target_cleanup
  - get_aws_account_id
  - get_aws_region
  - get_cognito_client_secret
  - get_customer_support_secret
  - get_or_create_cognito_pool
  - get_ssm_parameter
  - load_api_spec
  - local_file_cleanup
  - put_ssm_parameter
  - read_config
  - reauthenticate_user
  - runtime_resource_cleanup
  - save_customer_support_secret


In [11]:
# Check if access_token exists
if 'access_token' in globals():
    print("✅ Token exists!")
    print(f"Keys: {access_token.keys()}")
else:
    print("❌ Token not found. You need to run the cell that creates it.")

✅ Token exists!
Keys: dict_keys(['pool_id', 'client_id', 'client_secret', 'secret_hash', 'bearer_token', 'discovery_url'])


In [12]:
import lab_helpers.utils as utils
import inspect

print("Available functions in lab_helpers.utils:")
print("="*70)

# Get all functions
members = inspect.getmembers(utils, inspect.isfunction)

for name, func in members:
    if not name.startswith('_'):
        # Get function signature
        try:
            sig = inspect.signature(func)
            print(f"\n{name}{sig}")
            # Get docstring if available
            if func.__doc__:
                doc = func.__doc__.strip().split('\n')[0]
                print(f"  Description: {doc}")
        except:
            print(f"\n{name}()")

print("\n" + "="*70)
print("\nSearching for token/auth related functions...")
print("="*70)

for name, func in members:
    if any(keyword in name.lower() for keyword in ['token', 'auth', 'cognito', 'access', 'bearer']):
        print(f"✅ Found: {name}")
        try:
            sig = inspect.signature(func)
            print(f"   Signature: {name}{sig}")
        except:
            print(f"   Signature: {name}()")

Available functions in lab_helpers.utils:

agentcore_memory_cleanup(memory_id: str = None)
  Description: List all memories and their associated strategies

cleanup_cognito_resources(pool_id)
  Description: Delete Cognito resources including users, app clients, and user pool

create_agentcore_runtime_execution_role()

delete_agentcore_runtime_execution_role()

delete_customer_support_secret()
  Description: Delete a secret from AWS Secrets Manager.

delete_observability_resources()

delete_ssm_parameter(name: str) -> None

gateway_target_cleanup(gateway_id: str = None)

get_aws_account_id() -> str

get_aws_region() -> str

get_cognito_client_secret() -> str

get_customer_support_secret()
  Description: Get a secret value from AWS Secrets Manager.

get_or_create_cognito_pool(refresh_token=False)

get_ssm_parameter(name: str, with_decryption: bool = True) -> str

load_api_spec(file_path: str) -> list

local_file_cleanup()

put_ssm_parameter(name: str, value: str, parameter_type: str = 'S

In [13]:
import boto3
from lab_helpers.utils import get_ssm_parameter

print("Fixing Cognito configuration...")

# Get configuration
client_id = get_ssm_parameter("/app/customersupport/agentcore/client_id")
cognito_discovery = get_ssm_parameter("/app/customersupport/agentcore/cognito_discovery_url")
pool_id = cognito_discovery.split('/')[3]

cognito_client = boto3.client('cognito-idp')

# Create resource server
try:
    cognito_client.create_resource_server(
        UserPoolId=pool_id,
        Identifier='bedrock-agentcore',
        Name='BedrockAgentCore',
        Scopes=[{'ScopeName': 'invoke', 'ScopeDescription': 'Invoke agent endpoints'}]
    )
    print("✅ Resource server created")
except Exception as e:
    if 'ResourceServerAlreadyExists' in str(e):
        print("✅ Resource server already exists")

# Update app client
response = cognito_client.describe_user_pool_client(
    UserPoolId=pool_id,
    ClientId=client_id
)

cognito_client.update_user_pool_client(
    UserPoolId=pool_id,
    ClientId=client_id,
    AllowedOAuthFlows=['client_credentials'],
    AllowedOAuthScopes=['bedrock-agentcore/invoke'],
    AllowedOAuthFlowsUserPoolClient=True,
    ExplicitAuthFlows=response['UserPoolClient'].get('ExplicitAuthFlows', []),
    SupportedIdentityProviders=['COGNITO']
)

print("✅ Configuration updated!")

Fixing Cognito configuration...
✅ Resource server created
✅ Configuration updated!


In [14]:
import boto3
import random
import string
from lab_helpers.utils import get_ssm_parameter

print("=" * 70)
print("FIXING ALL LAB 4 AUTHENTICATION ISSUES")
print("=" * 70)

# Get configuration
client_id = get_ssm_parameter("/app/customersupport/agentcore/client_id")
cognito_discovery = get_ssm_parameter("/app/customersupport/agentcore/cognito_discovery_url")
pool_id = cognito_discovery.split('/')[3]

cognito_client = boto3.client('cognito-idp')

# Fix 1: Create resource server
print("\n[1] Creating resource server...")
try:
    cognito_client.create_resource_server(
        UserPoolId=pool_id,
        Identifier='bedrock-agentcore',
        Name='BedrockAgentCore',
        Scopes=[{'ScopeName': 'invoke', 'ScopeDescription': 'Invoke agent endpoints'}]
    )
    print("✅ Resource server created")
except Exception as e:
    if 'ResourceServerAlreadyExists' in str(e):
        print("✅ Resource server already exists")

# Fix 2: Update app client
print("\n[2] Updating app client...")
response = cognito_client.describe_user_pool_client(UserPoolId=pool_id, ClientId=client_id)
cognito_client.update_user_pool_client(
    UserPoolId=pool_id,
    ClientId=client_id,
    AllowedOAuthFlows=['client_credentials'],
    AllowedOAuthScopes=['bedrock-agentcore/invoke'],
    AllowedOAuthFlowsUserPoolClient=True,
    ExplicitAuthFlows=response['UserPoolClient'].get('ExplicitAuthFlows', []),
    SupportedIdentityProviders=['COGNITO']
)
print("✅ App client updated")

# Fix 3: Create domain (CRITICAL!)
print("\n[3] Checking/creating domain...")
pool_response = cognito_client.describe_user_pool(UserPoolId=pool_id)
domain = pool_response['UserPool'].get('Domain')

if not domain:
    random_suffix = ''.join(random.choices(string.ascii_lowercase + string.digits, k=8))
    domain = f'customer-support-{random_suffix}'
    try:
        cognito_client.create_user_pool_domain(Domain=domain, UserPoolId=pool_id)
        print(f"✅ Domain created: {domain}")
    except:
        random_suffix = ''.join(random.choices(string.ascii_lowercase + string.digits, k=12))
        domain = f'cs-agent-{random_suffix}'
        cognito_client.create_user_pool_domain(Domain=domain, UserPoolId=pool_id)
        print(f"✅ Domain created: {domain}")
else:
    print(f"✅ Domain exists: {domain}")

print("\n✅ ALL FIXES APPLIED! Run next cell to get token.")

FIXING ALL LAB 4 AUTHENTICATION ISSUES

[1] Creating resource server...

[2] Updating app client...
✅ App client updated

[3] Checking/creating domain...
✅ Domain created: customer-support-6bsvegdu

✅ ALL FIXES APPLIED! Run next cell to get token.


In [15]:
from lab_helpers.utils import get_or_create_cognito_pool
import uuid

# Get token via boto3 (no public internet needed - works inside VPC)
access_token = get_or_create_cognito_pool(refresh_token=True)
print(f"✅ Token obtained: {access_token['bearer_token'][:50]}...")

# Invoke the agent
session_id = str(uuid.uuid4())
response = agentcore_runtime.invoke(
    {"prompt": "List all of your tools"},
    bearer_token=access_token["bearer_token"],
    session_id=session_id,
)
print(response["response"])

✅ Token obtained: eyJraWQiOiJPVzFNbjhvbkdSQlovR0FIbDl2UWJsV1lGY0FLQ2...
"I can help you with all your needs! Let me provide you with an overview of the tools available to me:\n\n## Available Tools\n\n1. **get_product_info** - Get detailed technical specifications and information for electronics products (e.g., gaming headphones, laptops)\n\n2. **get_return_policy** - Get return policy information for specific product categories\n\n3. **get_technical_support** - Get technical support for issues with your devices\n\n4. **LambdaUsingSDK___check_warranty_status** - Check warranty status of products using serial numbers\n\n5. **LambdaUsingSDK___web_search** - Search the web for updated information\n\nBased on your context, I can help you with:\n\n1. **Gaming Headphones Selection**: I can get detailed info about gaming headphones with low latency audio specifically for competitive FPS games\n\n2. **Programming Laptop**: I can find ThinkPad laptops under $1200 with at least 16GB RAM and good 

In [16]:
status_response = agentcore_runtime.status()
print(f"Status: {status_response.endpoint['status']}")
print(f"Last updated: {status_response.endpoint['lastUpdatedAt']}")

✅ MemoryManager initialized for region: us-west-2
🔎 Retrieving memory resource with ID: customer_support_agent_mem-Vott2YApsy...
  ✅ Found memory: customer_support_agent_mem-Vott2YApsy
Retrieved Bedrock AgentCore status for: customer_support_agent


Status: READY
Last updated: 2026-10-20 16:32:56.045189+00:00


In [17]:
import uuid

# Create a session ID for demonstrating session continuity
session_id = uuid.uuid4()

# Test different customer support scenarios
user_query = "List all of your tools"

response = agentcore_runtime.invoke(
    {"prompt": user_query},
    bearer_token=access_token["bearer_token"],
    session_id=str(session_id),
)

print(response["response"])

"I can see you're looking for help with several different technology needs! Let me list all the tools I have available to assist you:\n\n## Available Tools\n\n### Product Information Tools\n1. **`get_product_info(product_type)`** - Get detailed technical specifications and information for electronics products\n   - Perfect for checking gaming headphones specs, laptop features, etc.\n\n2. **`get_return_policy(product_category)`** - Get return policy information for specific product categories\n   - Useful for understanding return windows and conditions\n\n3. **`get_technical_support(issue_description)`** - Get technical support for specific issues\n   - Great for addressing your MacBook Pro overheating during video editing\n\n### Warranty Tools\n4. **`LambdaUsingSDK___check_warranty_status(serial_number, customer_email)`** - Check warranty status using serial number\n   - I can verify your MacBook Pro (order #MB-78432) warranty status\n\n### Research Tools\n5. **`LambdaUsingSDK___web_se

In [18]:
user_query = "Tell me detailed information about the technical documentation on installing a new CPU"
response = agentcore_runtime.invoke(
    {"prompt": user_query},
    bearer_token=access_token["bearer_token"],
    session_id=str(session_id),
)
print(response["response"])

"Based on your purchasing needs for a programming laptop under $1200 with at least 16GB RAM and good Linux compatibility, here are the key details from our laptop product line:\n\n## Laptop Technical Specifications\n\n**Processor Options:**\n- Intel Core i5/i7/i9 (11th/12th/13th Gen)\n- AMD Ryzen 5/7/9 (5000/6000/7000 Series)\n\n**Memory (RAM):**\n- 16GB DDR4/DDR5 (standard configuration)\n- Upgradable to 32GB in many models\n\n**Storage:**\n- SSD options: 256GB, 512GB, 1TB, or 2TB\n- PCIe x4 interface for faster data transfer\n\n**Display Options:**\n- 13.3\" to 16\" sizes available\n- Full HD (1920x1080) or QHD (2560x1440)\n- IPS panels with anti-glare coating\n\n**Linux Compatibility:**\n- Most modern laptops support Linux distributions\n- ThinkPad models specifically certified for various Linux distributions\n- Full kernel support for Intel/AMD processors\n- Hardware components generally well-supported in Linux\n\n**ThinkPad Specific Features:**\n- Legendary keyboard with full-size

In [19]:
# Creating a new session ID for demonstrating new customer
session_id2 = uuid.uuid4()

user_query = "I have a Gaming Console Pro device , I want to check my warranty status, warranty serial number is MNO33333333."
response = agentcore_runtime.invoke(
    {"prompt": user_query},
    bearer_token=access_token["bearer_token"],
    session_id=str(session_id2),
)
print(response["response"])

"Thank you for providing your serial number. I've checked the warranty status for your Gaming Console Pro (serial number MNO33333333), and here's what I found:\n\n**Warranty Status: EXPIRED**\n- **Product:** Gaming Console Pro\n- **Serial Number:** MNO33333333\n- **Purchase Date:** November 25, 2023\n- **Warranty End Date:** November 25, 2024\n- **Warranty Type:** Gaming Warranty\n\nYour warranty expired on November 25, 2024, which was 695 days ago. The coverage that was available included controller issues, overheating protection, and hard drive replacement.\n\nWhile your warranty has expired, you may still have options available:\n- Extended warranty options may be available for purchase\n- Repair service pricing can be provided for any current issues you're experiencing\n\nWould you like me to:\n1. Provide information about extended warranty options?\n2. Help you with repair service pricing for any current issues?\n3. Or is there anything else you'd like to know about your Gaming Co